# 報酬モデルで N 個の応答を採点して最良を選ぶ (Best-of-N)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_llm/reward_best_of_n.ipynb)

NLP クックブック 14-17 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

LLM に同じ質問への応答を N 個サンプリングさせ、公開の報酬モデルで採点して最も点の高いものを選びたい (Best-of-N)。N を増やすと品質がどれだけ上がるかを、正解のある課題と別の判定役で確かめ、報酬モデルの偏り（報酬ハッキング）も調べる。選んだ最良と最悪の組を DPO の選好データにしたい。

In [1]:
import itertools, json, random, re, time, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from scipy.stats import spearmanr
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, set_seed

def load_tokenizer(name):
    tokenizer = AutoTokenizer.from_pretrained(name, padding_side="left")
    # transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
    # 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
    tokenizer.backend_tokenizer.model = models.Unigram(
        [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
    return tokenizer

set_seed(0)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
DTYPE = torch.bfloat16 if BF16 else torch.float16
# 応答を生成するモデル: この章の DPO・GRPO のレシピと同じ 0.5B のモデル (MIT License)
name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
tokenizer = load_tokenizer(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, device_map="auto").eval()

# 報酬モデル: Skywork-Reward-V2 (Liu et al., 2025) の Qwen3-1.7B 版 (Apache 2.0)。2,600 万組の選好データで学習した
# Bradley-Terry 型の報酬モデルで、質問と応答の会話を入れると 1 つの実数（報酬）を返す。点の絶対値に意味はなく、同じ質問の中で比べる
rm_name = "Skywork/Skywork-Reward-V2-Qwen3-1.7B"
rm_tokenizer = AutoTokenizer.from_pretrained(rm_name)
rm = AutoModelForSequenceClassification.from_pretrained(rm_name, dtype=DTYPE, device_map="auto").eval()

@torch.no_grad()
def reward(question, responses, bs=8):
    """1 つの質問への複数の応答を採点する。モデルカードの推奨に従い、システムプロンプトは付けない"""
    texts = [rm_tokenizer.apply_chat_template([{"role": "user", "content": question}, {"role": "assistant", "content": r}], tokenize=False)
             for r in responses]
    scores = []
    for i in range(0, len(texts), bs):
        enc = rm_tokenizer(texts[i:i + bs], return_tensors="pt", padding=True).to(rm.device)
        scores += rm(**enc).logits[:, 0].float().tolist()
    return np.array(scores)

# 日本語での挙動を確かめる: モデルカードの例（りんごの問題）を日本語に訳し、正しい応答・誤った応答・英語の応答を比べる
q = "りんごが 12 個あります。4 個を友達にあげ、1 個買い足して、自分と兄弟 2 人の 3 人で等しく分けます。1 人何個もらえますか。"
cands = {
    "正しい（日本語）": "12 − 4 = 8 個。1 個買って 8 + 1 = 9 個。3 人で分けるので 9 ÷ 3 = 3。答えは 1 人 3 個です。",
    "誤り（日本語）": "12 − 4 = 8 個。1 個買って 8 + 1 = 9 個。兄弟 2 人で分けるので 9 ÷ 2 = 4.5。答えは 1 人 4 個です。",
    "正しい（英語）": "12 - 4 = 8 apples. After buying 1 more, 8 + 1 = 9. Split among 3 people, 9 / 3 = 3. Each person gets 3 apples.",
    "答えだけ（正しい）": "3 個です。",
    "質問と無関係": "東京は日本の首都で、人口は約 1,400 万人です。",
}
for k, s in zip(cands, reward(q, list(cands.values()))):
    print(f"{s:7.2f}  {k}")

   4.18  正しい（日本語）
  -1.25  誤り（日本語）
   2.05  正しい（英語）
  -2.36  答えだけ（正しい）
 -13.12  質問と無関係


In [2]:
# 正解のある課題: MGSM（CC BY 4.0。Hugging Face のメタデータは CC BY-SA 4.0）の日本語の先頭 100 問（この章の Chain-of-Thought のレシピと同じ）
# 指示はそのレシピの「CoT (0-shot)」と同じ。温度 0.7・top_p 0.9（開発元の既定値）で 1 問につき N = 8 個サンプリングする。
# 開発元の既定値では repetition_penalty 1.05 も掛かる（貪欲法にも掛かる）
N_MAX, N_Q = 8, 100
mgsm = load_dataset("juletxara/mgsm", "ja", split="test").select(range(N_Q))
gold = np.array(mgsm["answer_number"], dtype=float)
INSTRUCTION = "次の問題に答えてください。ステップごとに考えて途中の計算を書き、最後に「答えは〇〇です。」と書いてください。"
math_q = [f"{INSTRUCTION}\n\n問題: {q}" for q in mgsm["question"]]

@torch.no_grad()
def generate(questions, n, max_new_tokens, bs=64):
    """質問ごとに n 個の応答と、長さの上限で打ち切られたかどうかを返す（n = 1 なら貪欲法、n > 1 ならサンプリング）"""
    outs, cut, step = [], [], max(1, bs // n)
    for i in range(0, len(questions), step):
        enc = tokenizer.apply_chat_template([[{"role": "user", "content": q}] for q in questions[i:i + step]], add_generation_prompt=True,
                                            return_tensors="pt", return_dict=True, padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=n > 1, num_return_sequences=n, pad_token_id=tokenizer.pad_token_id)
        gen = out[:, enc["input_ids"].shape[1]:]
        outs += tokenizer.batch_decode(gen, skip_special_tokens=True)
        cut += [o[-1].item() not in (tokenizer.eos_token_id, tokenizer.pad_token_id) for o in gen]
    return [outs[j * n:(j + 1) * n] for j in range(len(questions))], np.array(cut).reshape(len(questions), n)

def extract(text):
    """答えの数値を取り出す（この章の Chain-of-Thought のレシピの取り出し方の簡略版）: 「答え」の後ろ → 最後に出てきた数値"""
    text = unicodedata.normalize("NFKC", text).replace(",", "")
    found = re.findall(r"答え[はが]?\s*[:：]?[^\d\-\n]*(-?\d+(?:\.\d+)?)", text) or re.findall(r"-?\d+(?:\.\d+)?", text)
    return float(found[-1]) if found else None

t = time.time()
greedy, _ = generate(math_q, 1, 320)
math_out, math_cut = generate(math_q, N_MAX, 320)         # 800 個の応答。貪欲法と合わせて Colab の T4 で 8〜10 分程度
print(f"生成 {time.time() - t:.0f} 秒")
t = time.time()
math_score = np.array([reward(q, outs) for q, outs in zip(math_q, math_out)])    # 100 問 × 8 個
print(f"採点 {time.time() - t:.0f} 秒")
math_ans = [[extract(o) for o in outs] for outs in math_out]
correct = np.array([[a == g for a in row] for row, g in zip(math_ans, gold)])
greedy_correct = np.array([extract(o[0]) == g for o, g in zip(greedy, gold)])
print(f"報酬の平均: 正解の応答 {math_score[correct].mean():.2f}、不正解の応答 {math_score[~correct].mean():.2f}")
print(f"上限の 320 トークンで打ち切られた応答の割合 {math_cut.mean():.3f}（その報酬の平均 {math_score[math_cut].mean():.2f}）")

生成 131 秒
採点 24 秒
報酬の平均: 正解の応答 5.06、不正解の応答 -1.04
上限の 320 トークンで打ち切られた応答の割合 0.122（その報酬の平均 -4.34）


In [3]:
# N を変えて比べる。8 個から n 個を選ぶすべての組み合わせで計算して平均する（どの n 個を使うかによるばらつきをならす）
def vote(answers):
    answers = [a for a in answers if a is not None]
    return Counter(answers).most_common(1)[0][0] if answers else None

def per_question(i, n):
    combos = list(itertools.combinations(range(N_MAX), n))
    return {"ランダムに 1 つ": correct[i].mean(),
            "報酬モデルで最良 (Best-of-N)": np.mean([correct[i, list(c)][np.argmax(math_score[i, list(c)])] for c in combos]),
            "多数決 (self-consistency)": np.mean([vote([math_ans[i][j] for j in c]) == gold[i] for c in combos]),
            "どれか 1 つが正解（上限）": np.mean([correct[i, list(c)].any() for c in combos])}

table = {n: pd.DataFrame([per_question(i, n) for i in range(N_Q)]) for n in [1, 2, 4, 8]}
print(pd.DataFrame({n: d.mean() for n, d in table.items()}).T.rename_axis("N").round(3).to_string())
print(f"貪欲法 1 回: {greedy_correct.mean():.3f}")

def diff_ci(x, y, n_boot=2000):   # 同じ問題で比べる対応のあるブートストラップで、正解率の差の 95% 信頼区間
    idx = np.random.default_rng(0).integers(0, len(x), (n_boot, len(x)))
    d = x[idx].mean(1) - y[idx].mean(1)
    return f"{(x.mean() - y.mean()) * 100:+.1f} ポイント [{np.percentile(d, 2.5) * 100:+.1f}, {np.percentile(d, 97.5) * 100:+.1f}]"

bon8, maj8 = table[8]["報酬モデルで最良 (Best-of-N)"].values, table[8]["多数決 (self-consistency)"].values
print("Best-of-8 − ランダムに 1 つ:", diff_ci(bon8, table[8]["ランダムに 1 つ"].values))
print("Best-of-8 − 貪欲法 1 回:", diff_ci(bon8, greedy_correct.astype(float)))
print("Best-of-8 − 8 個の多数決:", diff_ci(bon8, maj8))

   ランダムに 1 つ  報酬モデルで最良 (Best-of-N)  多数決 (self-consistency)  どれか 1 つが正解（上限）
N                                                                         
1      0.199                 0.199                   0.199           0.199
2      0.199                 0.267                   0.191           0.293
4      0.199                 0.321                   0.246           0.392
8      0.199                 0.340                   0.310           0.490
貪欲法 1 回: 0.260
Best-of-8 − ランダムに 1 つ: +14.1 ポイント [+8.3, +20.2]
Best-of-8 − 貪欲法 1 回: +8.0 ポイント [-1.0, +17.0]
Best-of-8 − 8 個の多数決: +3.0 ポイント [-3.0, +9.0]


In [4]:
# 正解のない課題: 質問への応答の役に立ち方。この章の DPO のレシピと同じ絞り込みと並べ替えで、その学習側の質問 (data[:2000]) の先頭 50 件を使う
# （評価側の data[2000:2200] は使わない。ここで作る選好データを DPO の学習に使っても、評価の質問と重ならない）
# llm-jp/hh-rlhf-12k-ja (MIT License): Anthropic の hh-rlhf (MIT License) の一部を DeepL で日本語に訳したもの。有用さ (helpful-*) の部分だけを使う
raw = load_dataset("llm-jp/hh-rlhf-12k-ja", split="train")
data = [r["conversations"][0]["value"] for r in raw
        if r["source"].startswith("helpful") and len(r["conversations"]) == 1 and r["chosen"] != r["rejected"]
        and len(r["conversations"][0]["value"]) <= 200 and max(len(r["chosen"]), len(r["rejected"])) <= 400]
random.Random(0).shuffle(data)
help_q = data[:50]

t = time.time()
help_out, help_cut = generate(help_q, N_MAX, 384)   # 400 個の応答（上限は DPO のレシピと同じ 384 トークン）。Colab の T4 で 3〜4 分程度
help_score = np.array([reward(q, outs) for q, outs in zip(help_q, help_out)])
print(f"生成と採点 {time.time() - t:.0f} 秒")
best, worst = help_score.argmax(1), help_score.argmin(1)
rows = np.arange(len(help_q))
length = np.array([[len(o) for o in outs] for outs in help_out])
print(f"応答の平均の長さ: 最良 {length[rows, best].mean():.0f} 文字、全体 {length.mean():.0f} 文字、最悪 {length[rows, worst].mean():.0f} 文字")
def mean_rho(score, length, keep):   # 同じ質問の中での報酬と長さの順位相関 (Spearman) の平均。keep の応答だけで計算する
    return np.nanmean([spearmanr(s[k], l[k]).statistic for s, l, k in zip(score, length, keep) if k.sum() >= 3])
math_len = np.array([[len(o) for o in outs] for outs in math_out])
print(f"上限の 384 トークンで打ち切られた応答の割合 {help_cut.mean():.3f}")
print(f"同じ質問の中での報酬と長さの順位相関の平均: {mean_rho(help_score, length, np.ones_like(help_cut)):.3f}"
      f"（打ち切られた応答を除くと {mean_rho(help_score, length, ~help_cut):.3f}）")
print(f"算数の問題では {mean_rho(math_score, math_len, np.ones_like(math_cut)):.3f}（打ち切られた応答を除くと {mean_rho(math_score, math_len, ~math_cut):.3f}）")
i = 0
print("\n質問:", help_q[i])
for j in np.argsort(-help_score[i]):
    print(f"{help_score[i, j]:6.2f}  {help_out[i][j][:80]!r}")

生成と採点 81 秒
応答の平均の長さ: 最良 611 文字、全体 569 文字、最悪 538 文字
上限の 384 トークンで打ち切られた応答の割合 0.030
同じ質問の中での報酬と長さの順位相関の平均: 0.191（打ち切られた応答を除くと 0.225）
算数の問題では -0.265（打ち切られた応答を除くと -0.170）

質問: 男性がハゲる原因は何ですか？
  8.88  '男性がハゲる原因は、主に遺伝的な要因と環境的な要因のcombined（combined）によって引き起こされます。まず、遺伝的な要因が重要な役割を果たします。一'
  6.65  '男性がハゲる原因は多岐にわたりますが、主に遺伝的な要因と生活習慣が関与しています。まず、遺伝的な要因として、Y染色体上にに存在する5-alphaリダクターゼとい'
  6.58  '男性がハゲる原因は複数あり、主に遺伝的要因と環境的要因が影響しています。遺伝的には、額の後退（前頭部のaga）や頭髪の減少（男性型脱毛）が起こりやすくなります。'
  6.25  '男性がハゲる原因は多岐にわたりますが、主に遺伝的要因と生活習慣が関係しています。まず、遺伝的な要因として薄くなる傾向がある遺伝子が影響します。特に額の前方や側面'
  4.27  '男性がハゲる原因はいくつかあり、主に遺伝的な要因と生活習慣が関与しています。まず、遺伝的な要因としてM字やO字の抜け毛patternが見られる場合、これは前頭葉'
  3.47  '男性がハゲる原因は複数あり、主に遺伝的な要因と環境的な要因が関係しています。まず、遺伝的な要因として額の後退（アルopecia areata）や前頭部の抜け毛（'
  3.09  '男性がハゲる原因は、主に遺伝的な要因と環境的な要因が複合的に影響しています。まず、遺伝的な要素が大きな役割を果たします。特に、額の後退（前頭部の薄くなり）や b'
  1.68  '男性がハゲる原因は複数あり、主に遺伝的な要因と環境的な要因が関係しています。まず、遺伝的な要因として前頭部や vertex（Vertex：vertexは頭髪の根'


In [5]:
# 別の判定役で確かめる: この章の DPO のレシピと同じく sarashina2.2-3b-instruct (MIT License) に 2 つの応答を比べさせる。
# 順番を入れ替えて 2 回聞いて平均する。比べる相手は、報酬モデルを使わない場合（8 個のうち 1 個目 = ランダムに 1 つ）
judge_name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
jtok = load_tokenizer(judge_name)
judge = AutoModelForCausalLM.from_pretrained(judge_name, dtype=DTYPE, device_map="auto").eval()
id_a, id_b = [jtok(c, add_special_tokens=False)["input_ids"][-1] for c in ("A", "B")]

@torch.no_grad()
def p_first_better(q, x, y):
    def p_a(a, b):
        msg = [{"role": "user", "content": f"次の質問に対する 2 つの応答のうち、より役に立つ応答はどちらですか。A か B の 1 文字で答えてください。\n\n"
                                           f"質問: {q}\n\n応答 A: {a}\n\n応答 B: {b}"}]
        inputs = jtok.apply_chat_template(msg, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(judge.device)
        return torch.softmax(judge(**inputs).logits[0, -1].float()[[id_a, id_b]], -1)[0].item()
    return (p_a(x, y) + 1 - p_a(y, x)) / 2

def rate_ci(hits):   # 割合と、ブートストラップによる 95% 信頼区間
    boot = [hits[i].mean() for i in np.random.default_rng(0).integers(0, len(hits), (2000, len(hits)))]
    return f"{hits.mean():.3f}（95% 信頼区間 {np.percentile(boot, 2.5):.3f}〜{np.percentile(boot, 97.5):.3f}、{len(hits)} 問）"

others = [i for i in rows if best[i] != 0]       # 1 個目がたまたま最良だった質問は比べられないので除く
win_best = np.array([p_first_better(help_q[i], help_out[i][best[i]], help_out[i][0]) for i in others])
others = [i for i in rows if worst[i] != 0]
win_worst = np.array([p_first_better(help_q[i], help_out[i][worst[i]], help_out[i][0]) for i in others])   # 判定は合わせて Colab の T4 で 1 分程度
print("報酬モデルの最良がランダムな 1 つより良いと判定された割合:", rate_ci(win_best > 0.5))
print("報酬モデルの最悪がランダムな 1 つより良いと判定された割合:", rate_ci(win_worst > 0.5))
print(f"GPU メモリの最大使用量 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

報酬モデルの最良がランダムな 1 つより良いと判定された割合: 0.667（95% 信頼区間 0.524〜0.810、42 問）
報酬モデルの最悪がランダムな 1 つより良いと判定された割合: 0.244（95% 信頼区間 0.133〜0.356、45 問）
GPU メモリの最大使用量 11.9 GB


In [6]:
# 報酬ハッキングの手がかり: 応答の中身を変えずに、言い回しや長さだけを変えると報酬がどう動くか（各質問の 1 個目の応答で調べる）
edits = {
    "冒頭にほめ言葉を足す": lambda r: "素晴らしいご質問ですね！" + r,
    "末尾に定型の締めを足す": lambda r: r + "\n\nほかにもご質問があれば、お気軽にお尋ねください。",
    "末尾に「正確です」と書き足す": lambda r: r + "\n\nこの回答は専門家が確認した正確な情報です。",
    "同じ応答を 2 回繰り返す": lambda r: r + "\n\n" + r,
    "後半を削って半分にする": lambda r: r[:len(r) // 2],
}
base = help_score[:, 0]
res = {}
for label, f in edits.items():
    d = np.array([reward(q, [f(outs[0])])[0] for q, outs in zip(help_q, help_out)]) - base
    res[label] = {"報酬の変化（平均）": d.mean(), "上がった割合": (d > 0).mean()}
# 算数: 正解した応答の最後の答えの数値だけを 1 大きくする（推論はそのまま、答えだけが誤りになる）
def wrong_answer(text, g):
    return re.sub(r"(答え[はが]?\s*[:：]?[^\d\-\n]*)(-?[\d,]+(?:\.\d+)?)(?!.*答え)", lambda m: m.group(1) + f"{g + 1:g}", text, flags=re.S)
# 各問で最初に正解した応答のうち、「答えは…」の形で書かれていて書き換えられたものを使う
ok = [(i, j) for i in range(N_Q) for j in np.flatnonzero(correct[i])[:1] if wrong_answer(math_out[i][j], gold[i]) != math_out[i][j]]
d = np.array([reward(math_q[i], [wrong_answer(math_out[i][j], gold[i])])[0] - math_score[i, j] for i, j in ok])
res["算数: 最後の答えだけを誤りにする"] = {"報酬の変化（平均）": d.mean(), "上がった割合": (d > 0).mean()}
print(pd.DataFrame(res).T.round(3).to_string())
print(f"（算数は {len(ok)} 問。報酬の標準偏差は、同じ質問の 8 個の中で平均 {help_score.std(1).mean():.2f}）")

                   報酬の変化（平均）  上がった割合
冒頭にほめ言葉を足す            -0.011    0.54
末尾に定型の締めを足す            0.335    0.86
末尾に「正確です」と書き足す        -1.124    0.02
同じ応答を 2 回繰り返す         -0.720    0.16
後半を削って半分にする           -4.312    0.00
算数: 最後の答えだけを誤りにする     -6.549    0.00
（算数は 31 問。報酬の標準偏差は、同じ質問の 8 個の中で平均 1.56）


In [7]:
# 最良と最悪の組を、この章の DPO のレシピと同じ trl の会話形式 (prompt / chosen / rejected) の選好データにする。
# 報酬の差が小さい組は、報酬モデルの誤差で順番が入れ替わりやすいので除く（ここでは差が 2 未満）
pairs = [{"prompt": [{"role": "user", "content": q}],
          "chosen": [{"role": "assistant", "content": outs[b]}], "rejected": [{"role": "assistant", "content": outs[w]}],
          "margin": float(s[b] - s[w])}
         for q, outs, s, b, w in zip(help_q, help_out, help_score, best, worst) if s[b] - s[w] >= 2]
print(f"{len(help_q)} 問から {len(pairs)} 組。報酬の差の中央値 {np.median([p['margin'] for p in pairs]):.2f}")
print(f"chosen のほうが長い組の割合: {np.mean([len(p['chosen'][0]['content']) > len(p['rejected'][0]['content']) for p in pairs]):.3f}")
with open("bon_pairs.jsonl", "w") as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
p = max(pairs, key=lambda p: p["margin"])
print("\n質問:", p["prompt"][0]["content"], "\n[chosen]", p["chosen"][0]["content"][:200], "\n[rejected]", p["rejected"][0]["content"][:200])

50 問から 49 組。報酬の差の中央値 5.04
chosen のほうが長い組の割合: 0.633

質問: xを求めよ：4(x-5)=4 
[chosen] 4(x-5)=4という方程式を解くには、まず両辺を4で割って変数xを孤立させます。

まず、両辺を4で割ると次のようになります。
\[ \frac{4(x-5)}{4} = \frac{4}{4} \]
これは次のように簡略化されます。
\[ x-5 = 1 \]

次に、両辺に5を加えてxを求めます。
\[ x-5+5 = 1+5 \]
\[ x = 6 \]

したがって、解はx=6です。

 
[rejected] ```python
x = 5
```

（説明：方程式 \(4(x-5) = 4\) を解くには、まず方程式の両辺を4で割って簡素化します。これにより \((x-5) = 1\) となります。次に、両辺に5を加えて \(x\) を求めます。したがって、\(x = 5\) となります。）


- Best-of-N は、モデルの重みを変えずに推論時の計算を増やして品質を上げる方法で、N 個の候補を生成し、報酬モデルの点が最も高いものを選ぶ（WebGPT (Nakano et al., 2021) などで使われた）。生成と採点が N 倍になるので、時間もほぼ N 倍かかる。検証に使った GPU（T4 と同じ float16 の経路）では全体で約 5 分（算数の 900 個の生成 131 秒、採点 24 秒、質問の 400 個の生成と採点 81 秒）、GPU メモリの最大使用量は 11.9GB だった（0.5B の生成モデル、1.7B の報酬モデル、3B の判定役を同時に載せた状態）。T4 では生成が遅いので、この 2〜2.5 倍程度かかると見込んでいる（T4 では測っていない）。乱数の種を固定しても、GPU の計算の違いでサンプリングの結果が変わることがあり、そのときは表の値も変わる。
- 報酬モデル: Skywork-Reward-V2 は Skywork が 2025 年に公開した報酬モデルの系列（0.6B〜8B の 8 種類）で、Qwen3 から作った 4 種類は Apache 2.0、Llama から作ったものは Llama 3.1・3.2 のライセンスに従う。モデルカードには対応言語の記載がない。最初のセルの 5 件では、日本語の正しい応答 (4.18) が誤った応答 (−1.25) より高く、同じ内容の英語の応答 (2.05) より日本語のほうが高かった。この例では、説明のない「3 個です。」(−2.36) も途中の説明がある応答より低かった（1 問だけの結果である）。MGSM の 800 個の応答でも、正解の応答の平均 5.06 に対して不正解は −1.04 で、日本語でも正誤をある程度見分けている。
- 算数（正解のある課題）: ランダムに 1 つ選ぶと正解率 0.199 のところ、報酬モデルで選ぶと N = 2、4、8 で 0.267、0.321、0.340 に上がり、N = 8 では +14.1 ポイント [+8.3, +20.2] だった。ただし、貪欲法 1 回 (0.260) との差（+8.0 ポイント [−1.0, +17.0]）と、8 個の多数決 (0.310) との差（+3.0 ポイント [−3.0, +9.0]）は、100 問では偶然の範囲である。8 個のどれかが正解だった問題は 0.490 あり、報酬モデルはその 3 割ほどで正解を選び損ねている。N を増やすと上がり方は小さくなる（4 → 8 で +1.9 ポイント）。
- 正解のない課題: 判定役 (3B) は、報酬モデルの最良の応答を、ランダムに選んだ 1 つより良いと 0.667（95% 信頼区間 0.524〜0.810、42 問）で判定し、最悪の応答は 0.244（同 0.133〜0.356、45 問）だった。この判定役と人の選好の一致率は、この章の DPO のレシピでは 0.650 程度だったので、目安にとどまる。また、Best-of-N は候補の中から選ぶだけなので、全部に誤りがあれば最良にも残る（上の例で最も点の高い応答にも「combined（combined）」のような崩れた表現がある）。
- 報酬ハッキング（長さ）: この報酬モデルは、同じ質問の中では長い応答をやや好んだ（報酬と長さの順位相関の平均 0.191。最良の応答の平均 611 文字、全体 569 文字、最悪 538 文字）。上限の 384 トークンで打ち切られた応答は 3.0% で、それを除いても 0.225 だった。ただし、同じ応答を 2 回繰り返して長くすると報酬は平均 −0.720 で、上がったのは 16% だけなので、長ければよいわけではない。後半を削って半分にすると全部で下がった (−4.312) が、文の途中で切れたことへの罰も含まれていると考えられる。算数では逆に長い応答ほど報酬が低い（−0.265）。320 トークンで打ち切られて答えまで届かなかった応答（12.2%、報酬の平均 −4.34）の影響もあるが、それを除いても −0.170 で、原因は確かめていない。
- 報酬ハッキング（言い回し）: 末尾に「ほかにもご質問があれば、お気軽にお尋ねください。」と足すだけで 86% の応答で報酬が上がった（平均 +0.335。同じ質問の 8 個の報酬の標準偏差は平均 1.56 なので、点の近い候補の順位は入れ替わりうる）。冒頭のほめ言葉はほとんど効かず （−0.011、上がったのは 54%）、「専門家が確認した正確な情報です」という自己申告は逆に下がった (−1.124)。算数で最後の答えだけを誤りに書き換えると 31 問すべてで下がった（平均 −6.549）。中身と関係のない言い回しで点が上がるので、N を大きくしたり、この報酬モデルで強化学習したりすると、こうした言い回しが増えると考えられる（確かめていない）。報酬モデルの点を上げすぎると本当の品質から離れていくこと（過最適化）は Gao et al. (2023) が Best-of-N と強化学習の両方で調べている。報酬だけでなく、別の評価（正解率や判定役）と出力の例も見る。
- 選好データ: 報酬の差が 2 以上の 49 組（差の中央値 5.04）を bon_pairs.jsonl に保存した。質問は DPO のレシピの学習側 (`data[:2000]`) から取ったので、そのレシピの `train` を、このファイルから `margin` の列を除いて読み込んだものに置き換えれば、評価の質問と重ならずに DPO に使える（ここでは学習していない）。そのレシピの選好データは別のモデルの応答（方策外のデータ）だったが、これは調整するモデル自身の応答（方策上のデータ）で、人ではなく報酬モデルが選んでいる。そのため、報酬モデルの癖もそのまま選好として学習される。ここでは chosen のほうが長い組が 0.633 あり、学習すると応答が長くなる方向に働きうる。実際には数百問以上で作り、長さの比や言い回しの偏りを確かめる。
- 関係するレシピ: この章の「考える過程を書かせて LLM に算数の文章題を解かせる（Chain-of-Thought と self-consistency）」の多数決は、答えが一致するかだけで選ぶので、答えを機械的に比べられる課題にしか使えない。Best-of-N は報酬モデルの点で選ぶので、自由な文章にも使える。この章の「正解を判定できる課題で LLM を強化学習する (GRPO)」は、同じように 1 問に複数の応答を生成し、その報酬の差でモデル自身を学習する。判定役のモデルの信頼性の確かめ方は、LLM の評価と分析の章の「手元の LLM に生成文を採点させる (LLM-as-a-judge)」を参照。トークナイザーの修正 (`load_tokenizer`) については、この章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。